# 4.4 Transfer Learning

使用 MobileNetV2 做 flower_photos 分類。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

## 1. 載入資料

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
data_dir = pathlib.Path(tf.keras.utils.get_file('flower_photos', origin=url, untar=True))
# Some environments extract the archive into an extra nested folder.
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'
img_size = (160, 160)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='training', seed=42, image_size=img_size, batch_size=batch_size)
val_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='validation', seed=42, image_size=img_size, batch_size=batch_size)
class_names = train_ds.class_names
num_classes = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 2. 建立預訓練模型

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 3. 訓練分類頭

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Transfer Learning Accuracy')
plt.show()

## 4. 評估

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print({'val_loss': val_loss, 'val_accuracy': val_acc})

## 5. 套用自己的資料

換成自己的資料時，通常只需要替換圖片資料夾、調整 `img_size`、確認類別數，並保留對應 backbone 的 preprocessing function。若更換 backbone，前處理方式也要一起替換。


## 6. 小結

Transfer learning 先把預訓練模型當成 feature extractor，再訓練新的分類頭，是小型影像資料集最實用的 baseline。
